In [12]:
import pandas as pd
import numpy as np

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:

final_clv = pd.read_csv('../data/processed/clv_evaluated.csv', index_col=0)

print(final_clv.head())
print(final_clv.shape)

            frequency  recency      T  monetary_value  probability_alive  \
CustomerID                                                                 
12347.0           6.0    365.0  367.0      599.701667           0.996212   
12348.0           3.0    283.0  358.0      261.480000           0.982772   
12352.0           6.0    260.0  296.0      161.485000           0.992363   
12356.0           2.0    303.0  325.0      269.905000           0.984100   
12358.0           1.0    149.0  150.0      523.200000           0.959156   

            predicted_clv        error  absolute_error  percent_error  \
CustomerID                                                              
12347.0       2957.899780  2358.198114     2358.198114     392.573926   
12348.0        879.497234   618.017234      618.017234     235.453076   
12352.0       1273.603054  1112.118054     1112.118054     684.443520   
12356.0        737.574179   467.669179      467.669179     172.632169   
12358.0       1177.330656   6

In [ ]:

final_clv.to_csv('../data/processed/clv_evaluated.csv')

print("✓ Saved to: ../data/processed/clv_evaluated.csv")
print(f"✓ Columns: {list(final_clv.columns)}")
print(f"✓ Rows: {len(final_clv)}")

✓ Saved to: ../data/processed/clv_evaluated.csv
✓ Columns: ['frequency', 'recency', 'T', 'monetary_value', 'probability_alive', 'predicted_clv', 'error', 'absolute_error', 'percent_error', 'predicted_clv_calibrated', 'clv_segment']
✓ Rows: 4362


In [ ]:


X = final_clv[['frequency', 'recency', 'T']].copy()
y = final_clv['monetary_value'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train)} customers")
print(f"Test set: {len(X_test)} customers")

Training set: 3489 customers
Test set: 873 customers


In [ ]:

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("LINEAR REGRESSION RESULTS:")
print(f"MAE: ${mae_lr:.2f}")
print(f"R² Score: {r2_lr:.3f}")
print()

comparison_lr = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred_lr[:10],
    'Error': abs(y_test.values[:10] - y_pred_lr[:10])
})
print("First 10 predictions vs actual:")
print(comparison_lr)

LINEAR REGRESSION RESULTS:
MAE: $156.69
R² Score: 0.202

First 10 predictions vs actual:
       Actual   Predicted       Error
0  353.620000  380.202386   26.582386
1    0.000000   77.830874   77.830874
2  129.530000  323.261099  193.731099
3  207.500000  188.952551   18.547449
4  254.620000  427.398128  172.778128
5  384.400000  231.536456  152.863544
6  691.033333  251.862233  439.171100
7    0.000000   24.674954   24.674954
8  344.638333  452.161282  107.522949
9  371.945556  441.414043   69.468488


In [ ]:

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("RANDOM FOREST RESULTS:")
print(f"MAE: ${mae_rf:.2f}")
print(f"R² Score: {r2_rf:.3f}")
print()

print("Feature Importance:")
for feature, importance in zip(['frequency', 'recency', 'T'], rf_model.feature_importances_):
    print(f"  {feature}: {importance:.3f}")
print()
comparison_rf = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred_rf[:10],
    'Error': abs(y_test.values[:10] - y_pred_rf[:10])
})
print("First 10 predictions:")
print(comparison_rf)

RANDOM FOREST RESULTS:
MAE: $164.41
R² Score: -0.156

Feature Importance:
  frequency: 0.190
  recency: 0.500
  T: 0.310

First 10 predictions:
       Actual   Predicted       Error
0  353.620000  565.829501  212.209501
1    0.000000    0.000000    0.000000
2  129.530000  436.881402  307.351402
3  207.500000  338.134810  130.634810
4  254.620000  417.516714  162.896714
5  384.400000  202.520906  181.879094
6  691.033333  539.199944  151.833389
7    0.000000    0.000000    0.000000
8  344.638333  235.667366  108.970968
9  371.945556  441.743593   69.798038


In [19]:
!pip install xgboost 

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [20]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])

0

In [21]:
from xgboost import XGBRegressor
print("✓ XGBoost imported")

✓ XGBoost imported


In [ ]:

xgb_model = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print("XGBOOST RESULTS:")
print(f"MAE: ${mae_xgb:.2f}")
print(f"R² Score: {r2_xgb:.3f}")
print()

print("Feature Importance:")
for feature, importance in zip(['frequency', 'recency', 'T'], xgb_model.feature_importances_):
    print(f"  {feature}: {importance:.3f}")
print()

comparison_xgb = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred_xgb[:10],
    'Error': abs(y_test.values[:10] - y_pred_xgb[:10])
})
print("First 10 predictions:")
print(comparison_xgb)

XGBOOST RESULTS:
MAE: $150.76
R² Score: -0.010

Feature Importance:
  frequency: 0.243
  recency: 0.568
  T: 0.189

First 10 predictions:
       Actual   Predicted       Error
0  353.620000  384.864319   31.244319
1    0.000000    1.390340    1.390340
2  129.530000  432.417877  302.887877
3  207.500000  240.439087   32.939087
4  254.620000  303.891266   49.271266
5  384.400000  275.797028  108.602972
6  691.033333  422.245178  268.788155
7    0.000000   -1.203540    1.203540
8  344.638333  318.790192   25.848142
9  371.945556  531.599060  159.653505


In [ ]:

final_clv['predicted_clv_xgb'] = xgb_model.predict(X)

final_clv['clv_segment_xgb'] = pd.cut(final_clv['predicted_clv_xgb'], 
                                         bins=3, 
                                         labels=['Low Value', 'Medium Value', 'High Value'])

print(final_clv['clv_segment_xgb'].value_counts())
print()
print(final_clv.groupby('clv_segment_xgb')['predicted_clv_xgb'].mean())

clv_segment_xgb
Low Value       4336
Medium Value      22
High Value         4
Name: count, dtype: int64

clv_segment_xgb
Low Value        205.373230
Medium Value    1751.093262
High Value      6927.640137
Name: predicted_clv_xgb, dtype: float32


C:\Users\admin\AppData\Local\Temp\ipykernel_4668\740228931.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(final_clv.groupby('clv_segment_xgb')['predicted_clv_xgb'].mean())


In [ ]:

final_clv.to_csv('../data/processed/clv_final_xgb.csv')

print("✓ Saved to: ../data/processed/clv_final_xgb.csv")
print(f"✓ Columns: {list(final_clv.columns)}")

✓ Saved to: ../data/processed/clv_final_xgb.csv
✓ Columns: ['frequency', 'recency', 'T', 'monetary_value', 'probability_alive', 'predicted_clv', 'error', 'absolute_error', 'percent_error', 'predicted_clv_calibrated', 'clv_segment', 'predicted_clv_xgb', 'clv_segment_xgb']
